In [1]:
from pathlib import Path
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'MPMs').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
import os
os.environ['JAX_PLATFORMS'] = 'cpu'
import importlib
import equinox as eqx
import jax.numpy as jnp
import jax
import diffrax as dfx
import numpy as np
import matplotlib.pyplot as plt

import MPMs.DATA_functions_01 as DF
import MPMs.UTIL_functions_01 as UF
import MPMs.ANA_functions_02 as AF
import MPMs.HELPER_functions_02 as HF
import MPMs.MPM_functions_07 as MF

importlib.reload(UF)
importlib.reload(DF)
importlib.reload(HF)
importlib.reload(AF);

# Generate ANN data

In [2]:
if "MM" in vars():
    del MM

#--- INPUTs ---#
base_path = f"{REPO_ROOT}/prol_model_scripts/"
model_path = f"{base_path}/MPM_146B.py"
#--------------#

MM = UF.import_module_from_path(model_path)
exp_dataframes = MM.get_exp_dfs()
aug_dataframes = MM.get_aug_dfs()
cross_valid_sets = MM.HF.get_valid_sets()
scalers, ALL_kwargs = MM.get_model_config()

CV_models = {}
for set_index in range(len(cross_valid_sets)):
    ALL_data = MM.preprocess_data(exp_dataframes,
                                     aug_dataframes,
                                     cross_valid_sets[set_index],
                                     scalers,augmented=True,verbose=False)
    
    ODE_kwargs, TRAIN_kwargs = ALL_kwargs
    ODE_kwargs["verbose"] = False
    model = MM.ODE(**ODE_kwargs)
    model = eqx.nn.inference_mode(model)

    weights_path = model_path.replace("model_scripts","model_data").replace(".py",f"/set_{set_index+1}_last.eqx")
    print(weights_path)
    model = eqx.tree_deserialise_leaves(weights_path, model)
    model = eqx.nn.inference_mode(model)
    CV_models[f"set_{set_index+1}"] = (model, ALL_data)
print("DONE")

#-------------------- CONFIG --------------------#
fba_type          : SRfba
base_flux_type    : linear
LIM_handler_type  : min
modify_ann_inp    : none
extra_metabolites : False
#------------------------------------------------#


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_1_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_2_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_3_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_4_last.eqx


...

/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_8_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_9_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_10_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_11_last.eqx


/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/set_12_last.eqx
DONE


In [3]:
import pandas as pd
importlib.reload(AF)

# output is: YY, Yt, V_out, yy, fba_obj

def normizer(a):
    a = jnp.abs(a)
    s = jnp.sum(a)+1e-8
    return a/s

key = jax.random.PRNGKey(0)

combs = []
for set_name, tmp in CV_models.items():
    tmp_model, tmp_data = tmp
    # predict with TRAIN data tmp_data[0]
    PRED = AF.NEW_get_predictions(tmp_model, tmp_data[0], key, mode="pred")
    inp = UF.squeeze(PRED["mod_ann_inp"])
    out = UF.squeeze(PRED["FBA_inp"])[:,1:] # get nX, nP, nM
    out = jax.vmap(normizer)(out)
    comb = jnp.concatenate([inp, out], axis=1)
    print(inp.shape,out.shape,comb.shape)
    comb = pd.DataFrame(comb)
    comb.columns = [f"var_{i+1}" for i in range(comb.columns.shape[0]-3)]+["nX","nP","nM"]
    comb["set"] = set_name
    combs.append(comb)
    # break
combs = pd.concat(combs)
save_path = model_path.replace("model_scripts","model_data").replace(".py",f"/CV_ann_fba_obj.csv")
combs.to_csv(save_path, index=False)
print(f"saved {model_path} model data to\n{save_path}")
print("done")

(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


...

(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)


(1155, 12) (1155, 3) (1155, 15)
saved /home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_scripts//MPM_146B.py model data to
/home/mgotsmy/code/260421_fba_hyb/FBA-Hyb/prol_model_data//MPM_146B/CV_ann_fba_obj.csv
done
